# 🚢 Titanic - Machine Learning from Disaster

## Phase 10: Competition Optimization, Robust Validation & Group-Aware Feature Engineering

### Objective

Our first competition submissions produced the following result:

| Model               | Local OOF Accuracy | Kaggle Public Score |
| ------------------- | -----------------: | ------------------: |
| Tuned Random Forest |             0.8541 |             0.75119 |
| Weighted Ensemble   |             0.8541 |             0.75119 |

This reveals an important problem.

Our local validation performance is substantially higher than our Kaggle leaderboard performance.

The difference is approximately:

`0.8541 - 0.75119 = 0.10291`

A gap of more than 10 percentage points suggests that improving the existing model through additional hyperparameter tuning may not be the best next step.

Instead, we need to investigate whether:

* Our validation strategy is too optimistic.
* Related passengers are appearing across training and validation folds.
* Some engineered features behave differently between train and test.
* Our models are learning unstable patterns.
* Family and ticket groups require more careful modeling.
* Simpler Titanic-specific rules generalize better than our current models.

---

# Phase 10 Strategy

This notebook will focus on five areas.

## 1. Validation Diagnostics

We will compare multiple validation strategies rather than trusting a single 5-fold split.

## 2. Group Structure

We will investigate passengers connected through:

* Surname.
* Family.
* Ticket.
* Traveling groups.

## 3. New Titanic-Specific Features

We will create features such as:

* `Surname`
* `FamilyID`
* `TicketGroupSize`
* `FamilyGroupSize`
* `Mother`
* `Child`
* `Woman`
* `Boy`
* `FarePerPerson`

## 4. Robust Model Comparison

We will compare simple and complex models under multiple validation seeds.

## 5. Submission Candidate Selection

Only models that remain strong under robust validation will be carried forward.

---

# Important Competition Principle

The objective is not to force the local validation score toward 1.000.

The objective is to build a validation framework that predicts unseen performance more reliably.

A model with:

`0.82 robust validation accuracy`

may be more useful than a model with:

`0.86 optimistic validation accuracy`

if the first model generalizes better to unseen passengers.

Our goal is:

> Improve leaderboard performance through better generalization, not through validation overfitting.


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import (
    StratifiedKFold,
    GroupKFold,
    cross_validate,
    cross_val_predict
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)

import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


# 2. Define Project Paths

Unlike the previous modeling notebooks, this phase requires access to the original passenger-level information.

Features such as surname and ticket groups are derived from:

* `Name`
* `Ticket`

Therefore, we will load the original Kaggle datasets from `data/raw/`.

We will create all Phase 10 features inside this notebook so that the exact transformation logic remains visible and reproducible.


In [3]:
raw_data_dir = Path(
    "../data/raw"
)

results_dir = Path(
    "../results"
)

submissions_dir = Path(
    "../submissions"
)

results_dir.mkdir(
    parents=True,
    exist_ok=True
)

submissions_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Directories created successfully.")

Directories created successfully.


In [4]:
train_raw = pd.read_csv(
    raw_data_dir / "train.csv"
)

test_raw = pd.read_csv(
    raw_data_dir / "test.csv"
)

print(
    f"Training shape: {train_raw.shape}"
)

print(
    f"Test shape: {test_raw.shape}"
)

train_raw.head()

Training shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 3. Combine Train and Test for Structural Feature Engineering

Some features depend on group frequency.

For example:

* How many passengers share the same ticket?
* How many passengers belong to the same surname-family group?

If we calculate these counts separately in the training and test datasets, the group sizes may be incomplete.

We will temporarily combine train and test to calculate structural features that do not use the `Survived` target.

This is acceptable because these features use only passenger information available at prediction time.

We will not use test survival labels, because they are unavailable.

Any future feature derived from `Survived` must be generated using strict out-of-fold logic and must never use unknown test labels.


In [5]:
train_raw = train_raw.copy()
test_raw = test_raw.copy()

train_raw["Dataset"] = "train"
test_raw["Dataset"] = "test"

test_raw["Survived"] = np.nan

combined = pd.concat(
    [
        train_raw,
        test_raw
    ],
    ignore_index=True
)

print(
    f"Combined shape: {combined.shape}"
)

Combined shape: (1309, 13)


# 4. Extract Passenger Title

Passenger titles provide information about:

* Sex.
* Age.
* Social status.
* Marital status.

Examples include:

* Mr
* Mrs
* Miss
* Master

Rare titles will be grouped into a smaller number of meaningful categories to reduce sparsity.


In [7]:
combined["Title"] = (
    combined["Name"]
    .str.extract(
        r",\s*([^.]*)\."
    )[0]
    .str.strip()
)


title_mapping = {

    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",

    "Lady": "Rare",
    "Countess": "Rare",
    "the Countess": "Rare",
    "Capt": "Rare",
    "Col": "Rare",
    "Don": "Rare",
    "Dr": "Rare",
    "Major": "Rare",
    "Rev": "Rare",
    "Sir": "Rare",
    "Jonkheer": "Rare",
    "Dona": "Rare"
}


combined["Title"] = (
    combined["Title"]
    .replace(
        title_mapping
    )
)


combined[
    "Title"
].value_counts()

Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64

# 5. Extract Passenger Surname

Passengers traveling as families often share a surname.

We will extract the surname from the `Name` column.

Surname alone is not sufficient to identify a family because unrelated passengers may share common surnames.

Later, we will combine surname with family size to construct a more specific family identifier.


In [8]:
combined["Surname"] = (

    combined["Name"]

    .str.split(",")

    .str[0]

    .str.strip()
)


combined[
    [
        "Name",
        "Surname"
    ]
].head()

,Name,Surname
0,"Braund, Mr. Owen Harris",Braund
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Cumings
2,"Heikkinen, Miss. Laina",Heikkinen
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Futrelle
4,"Allen, Mr. William Henry",Allen


# 6. Create Family Structure Features

The original dataset contains:

* `SibSp`: Number of siblings and spouses aboard.
* `Parch`: Number of parents and children aboard.

We will derive:

### FamilySize

`SibSp + Parch + 1`

The additional `1` represents the passenger.

### IsAlone

Indicates whether the passenger traveled without immediate family.

### FamilyCategory

Groups passengers into broad family-size categories.

These representations allow models to capture nonlinear relationships between family size and survival.


In [9]:
combined["FamilySize"] = (

    combined["SibSp"]

    +

    combined["Parch"]

    +

    1
)


combined["IsAlone"] = (

    combined["FamilySize"] == 1

).astype(int)


combined["FamilyCategory"] = pd.cut(

    combined["FamilySize"],

    bins=[
        0,
        1,
        4,
        7,
        np.inf
    ],

    labels=[
        "Alone",
        "Small",
        "Medium",
        "Large"
    ]
)

# 7. Create a Family Identifier

We will combine:

`Surname + FamilySize`

to create a `FamilyID`.

For example:

`Andersson_7`

This can help identify passengers likely to belong to the same family group.

Very small groups are less reliable as family identifiers.

Therefore, singleton families will be grouped into a generic category for categorical modeling.

The raw family identifier will still be retained separately for validation-group analysis.


In [10]:
combined["FamilyID_Raw"] = (

    combined["Surname"]

    +

    "_"

    +

    combined["FamilySize"]
    .astype(str)
)


family_counts = (

    combined[
        "FamilyID_Raw"
    ]
    .value_counts()
)


combined["FamilyID"] = (

    combined[
        "FamilyID_Raw"
    ]
)


combined.loc[

    combined[
        "FamilyID"
    ]
    .map(
        family_counts
    )
    <= 1,

    "FamilyID"

] = "Small_or_Single"

# 8. Create Ticket Group Features

Passengers sharing the same ticket often traveled together.

Ticket information can therefore represent groups that are not captured by family relationships alone.

We will create:

### TicketGroupSize

Number of passengers sharing the same ticket across the combined train and test passenger manifest.

### TicketPrefix

The non-numeric portion of the ticket.

### SharedTicket

Indicates whether the passenger shares a ticket with at least one other passenger.


In [11]:
combined["TicketClean"] = (

    combined["Ticket"]

    .astype(str)

    .str.upper()

    .str.replace(
        r"\s+",
        "",
        regex=True
    )
)


ticket_counts = (

    combined[
        "TicketClean"
    ]
    .value_counts()
)


combined["TicketGroupSize"] = (

    combined[
        "TicketClean"
    ]
    .map(
        ticket_counts
    )
)


combined["SharedTicket"] = (

    combined[
        "TicketGroupSize"
    ]
    > 1

).astype(int)


combined["TicketPrefix"] = (

    combined["Ticket"]

    .astype(str)

    .str.replace(
        r"\d",
        "",
        regex=True
    )

    .str.replace(
        r"[\s./]",
        "",
        regex=True
    )

    .str.upper()
)


combined["TicketPrefix"] = (

    combined[
        "TicketPrefix"
    ]
    .replace(
        "",
        "NONE"
    )
)

In [12]:
combined["CabinKnown"] = (

    combined[
        "Cabin"
    ]
    .notna()

).astype(int)


combined["Deck"] = (

    combined[
        "Cabin"
    ]

    .fillna(
        "Unknown"
    )

    .astype(str)

    .str[0]
)

# 9. Create Fare Features

A ticket fare may represent the total amount associated with multiple passengers sharing a booking.

Therefore, raw `Fare` may not represent the effective fare paid per passenger.

We will create:

### FarePerPerson

`Fare / TicketGroupSize`

### LogFare

A logarithmic transformation that reduces the influence of extremely large fare values.


In [13]:
combined["Fare"] = (

    combined[
        "Fare"
    ]
    .fillna(

        combined[
            "Fare"
        ]
        .median()
    )
)


combined["FarePerPerson"] = (

    combined["Fare"]

    /

    combined[
        "TicketGroupSize"
    ]
)


combined["LogFare"] = (

    np.log1p(

        combined[
            "Fare"
        ]
    )
)

# 10. Improve Age Imputation

Age is missing for a substantial number of passengers.

Using one global median ignores useful information.

Titles strongly correlate with age.

For example:

* `Master` generally represents boys.
* `Miss` includes girls and unmarried women.
* `Mrs` generally represents adult women.
* `Mr` generally represents adult men.

We will therefore calculate median age using:

`Title + Pclass`

and use these group-specific medians to fill missing ages.

If a group median is unavailable, we will fall back to the overall median age.


In [14]:
combined["AgeMissing"] = (

    combined[
        "Age"
    ]
    .isna()

).astype(int)


age_group_medians = (

    combined

    .groupby(
        [
            "Title",
            "Pclass"
        ]
    )[
        "Age"
    ]

    .transform(
        "median"
    )
)


combined["Age"] = (

    combined[
        "Age"
    ]

    .fillna(
        age_group_medians
    )

    .fillna(

        combined[
            "Age"
        ]
        .median()
    )
)

# 11. Create Titanic-Specific Demographic Features

Historical survival patterns on the Titanic were strongly related to sex and age.

Rather than relying only on raw `Sex` and `Age`, we will create explicit demographic indicators.

### Child

Passenger younger than 16.

### Woman

Female passenger aged 16 or older.

### Boy

Male passenger younger than 16.

### Mother

Adult female passenger traveling with at least one parent/child relation recorded through `Parch`.

These features provide interpretable representations of passenger demographic roles.


In [16]:
combined["Child"] = (

    combined[
        "Age"
    ]
    < 16

).astype(int)


combined["Woman"] = (

    (
        combined[
            "Sex"
        ]
        == "female"
    )

    &

    (
        combined[
            "Age"
        ]
        >= 16
    )

).astype(int)


combined["Boy"] = (

    (
        combined[
            "Sex"
        ]
        == "male"
    )

    &

    (
        combined[
            "Age"
        ]
        < 16
    )

).astype(int)


combined["Mother"] = (

    (
        combined[
            "Sex"
        ]
        == "female"
    )

    &

    (
        combined[
            "Age"
        ]
        > 18
    )

    &

    (
        combined[
            "Parch"
        ]
        > 0
    )

    &

    (
        combined[
            "Title"
        ]
        != "Miss"
    )

).astype(int)

In [17]:
combined["Sex_Pclass"] = (

    combined[
        "Sex"
    ]
    .astype(str)

    +

    "_"

    +

    combined[
        "Pclass"
    ]
    .astype(str)
)


combined["Age_Pclass"] = (

    combined[
        "Age"
    ]

    *

    combined[
        "Pclass"
    ]
)

In [18]:
train_opt = (

    combined[
        combined[
            "Dataset"
        ]
        == "train"
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


test_opt = (

    combined[
        combined[
            "Dataset"
        ]
        == "test"
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


train_opt[
    "Survived"
] = (

    train_opt[
        "Survived"
    ]
    .astype(int)
)


print(
    f"Optimized train shape: "
    f"{train_opt.shape}"
)


print(
    f"Optimized test shape: "
    f"{test_opt.shape}"
)

Optimized train shape: (891, 35)
Optimized test shape: (418, 35)


# 12. Define Competition Feature Set

We will use a carefully selected feature set containing:

* Original demographic information.
* Family structure.
* Ticket-group information.
* Cabin information.
* Fare information.
* Titanic-specific demographic indicators.
* Selected interactions.

We will initially exclude raw high-cardinality fields such as:

* Passenger name.
* Raw ticket number.
* Raw cabin number.
* Passenger ID.

`FamilyID` will initially be excluded from the predictive feature set because it is high-cardinality and may encourage memorization.

However, `FamilyID_Raw` will be extremely useful for group-aware validation.


In [19]:
features = [

    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",

    "Title",

    "FamilySize",
    "IsAlone",
    "FamilyCategory",

    "TicketGroupSize",
    "SharedTicket",
    "TicketPrefix",

    "CabinKnown",
    "Deck",

    "FarePerPerson",
    "LogFare",

    "AgeMissing",

    "Child",
    "Woman",
    "Boy",
    "Mother",

    "Sex_Pclass",
    "Age_Pclass"
]


X = train_opt[
    features
].copy()


y = train_opt[
    "Survived"
].copy()


print(
    f"Features selected: "
    f"{len(features)}"
)

Features selected: 25


In [20]:
categorical_features = [

    "Pclass",
    "Sex",
    "Embarked",
    "Title",
    "FamilyCategory",
    "TicketPrefix",
    "Deck",
    "Sex_Pclass"
]


numerical_features = [

    feature

    for feature
    in features

    if feature
    not in categorical_features
]


numeric_transformer = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(
                strategy="median"
            )
        ),

        (
            "scaler",

            StandardScaler()
        )
    ]
)


categorical_transformer = Pipeline(

    steps=[

        (
            "imputer",

            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",

            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor = ColumnTransformer(

    transformers=[

        (
            "numeric",

            numeric_transformer,

            numerical_features
        ),

        (
            "categorical",

            categorical_transformer,

            categorical_features
        )
    ]
)

# 13. Establish Titanic-Specific Rule Baselines

Before evaluating more machine learning models, we will establish several simple rule-based baselines.

These models require no training.

### Rule 1 — Predict All Women Survive

This is similar to the logic behind Kaggle's sample `gender_submission.csv`.

### Rule 2 — Women and Young Boys

Predict survival for:

* Female passengers.
* Boys younger than a selected age.

These baselines help us determine whether our machine learning models provide meaningful improvement over simple demographic rules.


In [21]:
gender_rule_predictions = (

    train_opt[
        "Sex"
    ]
    == "female"

).astype(int)


gender_rule_accuracy = (

    accuracy_score(

        y,

        gender_rule_predictions
    )
)


women_children_predictions = (

    (
        train_opt[
            "Sex"
        ]
        == "female"
    )

    |

    (
        train_opt[
            "Age"
        ]
        < 12
    )

).astype(int)


women_children_accuracy = (

    accuracy_score(

        y,

        women_children_predictions
    )
)


print(
    f"Women-only rule: "
    f"{gender_rule_accuracy:.4f}"
)


print(
    f"Women + children rule: "
    f"{women_children_accuracy:.4f}"
)

Women-only rule: 0.7868
Women + children rule: 0.7912


In [22]:
models = {

    "Logistic Regression":

        LogisticRegression(

            C=0.5,

            max_iter=3000,

            random_state=42
        ),


    "Random Forest":

        RandomForestClassifier(

            n_estimators=700,

            max_depth=5,

            min_samples_split=6,

            min_samples_leaf=3,

            max_features="sqrt",

            random_state=42,

            n_jobs=-1
        ),


    "Extra Trees":

        ExtraTreesClassifier(

            n_estimators=700,

            max_depth=6,

            min_samples_split=6,

            min_samples_leaf=3,

            max_features="sqrt",

            random_state=42,

            n_jobs=-1
        )
}

# 14. Evaluate Models Across Multiple Cross-Validation Seeds

Our previous modeling process relied heavily on one cross-validation configuration:

`random_state = 42`

On a dataset containing only 891 passengers, a model's score can depend significantly on how passengers are divided into folds.

We will therefore repeat Stratified 5-Fold Cross-Validation using multiple random seeds.

This allows us to evaluate:

* Average performance across different splits.
* Performance variability.
* Worst-case validation performance.
* Best-case validation performance.

A model that performs consistently across many splits is more trustworthy than one that performs exceptionally well on only one split.


In [23]:
cv_seeds = [

    7,
    21,
    42,
    77,
    101,
    202,
    404
]


robust_results = []


for model_name, model in (
    models.items()
):

    print(
        f"\nEvaluating: "
        f"{model_name}"
    )


    seed_scores = []


    for seed in cv_seeds:

        cv = StratifiedKFold(

            n_splits=5,

            shuffle=True,

            random_state=seed
        )


        pipeline = Pipeline(

            steps=[

                (
                    "preprocessor",
                    preprocessor
                ),

                (
                    "classifier",
                    model
                )
            ]
        )


        scores = cross_validate(

            pipeline,

            X,

            y,

            cv=cv,

            scoring="accuracy",

            n_jobs=-1
        )


        seed_scores.extend(

            scores[
                "test_score"
            ]
        )


    robust_results.append({

        "Model":
            model_name,

        "Mean Accuracy":
            np.mean(
                seed_scores
            ),

        "Std Accuracy":
            np.std(
                seed_scores
            ),

        "Min Accuracy":
            np.min(
                seed_scores
            ),

        "Max Accuracy":
            np.max(
                seed_scores
            )
    })


robust_results_df = (

    pd.DataFrame(
        robust_results
    )

    .sort_values(

        "Mean Accuracy",

        ascending=False
    )

    .reset_index(
        drop=True
    )
)


robust_results_df.round(4)


Evaluating: Logistic Regression

Evaluating: Random Forest

Evaluating: Extra Trees


,Model,Mean Accuracy,Std Accuracy,Min Accuracy,Max Accuracy
0,Random Forest,0.8331,0.0221,0.7809,0.8764
1,Extra Trees,0.8310,0.0201,0.7640,0.8596
2,Logistic Regression,0.8278,0.0219,0.7753,0.8708


# 15. Family-Aware Validation Diagnostic

Random cross-validation may place members of the same family in both:

* Training folds.
* Validation folds.

Because relatives often share:

* Surname.
* Ticket.
* Fare.
* Cabin.
* Survival circumstances.

this can make validation performance optimistic.

We will therefore perform a diagnostic using `GroupKFold`.

Passengers belonging to the same `FamilyID_Raw` group will remain in the same fold.

This is a deliberately stricter test.

It does not perfectly reproduce the Kaggle train-test split, but it helps reveal whether our models depend heavily on patterns shared between related passengers.


In [24]:
family_groups = (

    train_opt[
        "FamilyID_Raw"
    ]
)


group_cv = GroupKFold(

    n_splits=5
)


family_group_results = []


for model_name, model in (
    models.items()
):

    pipeline = Pipeline(

        steps=[

            (
                "preprocessor",
                preprocessor
            ),

            (
                "classifier",
                model
            )
        ]
    )


    scores = cross_validate(

        pipeline,

        X,

        y,

        groups=family_groups,

        cv=group_cv,

        scoring="accuracy",

        n_jobs=-1
    )


    family_group_results.append({

        "Model":
            model_name,

        "Family Group CV Mean":
            scores[
                "test_score"
            ].mean(),

        "Family Group CV Std":
            scores[
                "test_score"
            ].std(),

        "Family Group CV Min":
            scores[
                "test_score"
            ].min(),

        "Family Group CV Max":
            scores[
                "test_score"
            ].max()
    })


family_group_df = (

    pd.DataFrame(

        family_group_results
    )

    .sort_values(

        "Family Group CV Mean",

        ascending=False
    )
)


family_group_df.round(4)

,Model,Family Group CV Mean,Family Group CV Std,Family Group CV Min,Family Group CV Max
1,Random Forest,0.8305,0.0305,0.7753,0.8652
2,Extra Trees,0.8294,0.0205,0.7921,0.8547
0,Logistic Regression,0.8204,0.0246,0.7809,0.8492


In [25]:
validation_comparison = (

    robust_results_df

    .merge(

        family_group_df,

        on="Model",

        how="left"
    )
)


validation_comparison[

    "Random-vs-Group Gap"

] = (

    validation_comparison[
        "Mean Accuracy"
    ]

    -

    validation_comparison[
        "Family Group CV Mean"
    ]
)


validation_comparison.round(4)

,Model,Mean Accuracy,Std Accuracy,Min Accuracy,Max Accuracy,Family Group CV Mean,Family Group CV Std,Family Group CV Min,Family Group CV Max,Random-vs-Group Gap
0,Random Forest,0.8331,0.0221,0.7809,0.8764,0.8305,0.0305,0.7753,0.8652,0.0026
1,Extra Trees,0.8310,0.0201,0.7640,0.8596,0.8294,0.0205,0.7921,0.8547,0.0016
2,Logistic Regression,0.8278,0.0219,0.7753,0.8708,0.8204,0.0246,0.7809,0.8492,0.0074


# 16. Ticket-Aware Validation Diagnostic

Families are not the only passenger groups.

Unrelated passengers may also share a ticket and travel together.

We will therefore repeat group-aware validation using cleaned ticket numbers as the grouping variable.

All passengers sharing a ticket will remain in the same fold.

If performance drops substantially compared with random cross-validation, this suggests that shared-ticket information may be contributing to optimistic random-fold validation.


In [26]:
ticket_groups = (

    train_opt[
        "TicketClean"
    ]
)


ticket_group_cv = GroupKFold(

    n_splits=5
)


ticket_group_results = []


for model_name, model in (
    models.items()
):

    pipeline = Pipeline(

        steps=[

            (
                "preprocessor",
                preprocessor
            ),

            (
                "classifier",
                model
            )
        ]
    )


    scores = cross_validate(

        pipeline,

        X,

        y,

        groups=ticket_groups,

        cv=ticket_group_cv,

        scoring="accuracy",

        n_jobs=-1
    )


    ticket_group_results.append({

        "Model":
            model_name,

        "Ticket Group CV Mean":
            scores[
                "test_score"
            ].mean(),

        "Ticket Group CV Std":
            scores[
                "test_score"
            ].std()
    })


ticket_group_df = pd.DataFrame(

    ticket_group_results
)


ticket_group_df.round(4)

,Model,Ticket Group CV Mean,Ticket Group CV Std
0,Logistic Regression,0.8215,0.0293
1,Random Forest,0.8238,0.0229
2,Extra Trees,0.8215,0.0188


In [27]:
full_validation_comparison = (

    validation_comparison

    .merge(

        ticket_group_df,

        on="Model",

        how="left"
    )
)


full_validation_comparison[

    [

        "Model",

        "Mean Accuracy",

        "Std Accuracy",

        "Family Group CV Mean",

        "Ticket Group CV Mean",

        "Random-vs-Group Gap"
    ]

].round(4)

,Model,Mean Accuracy,Std Accuracy,Family Group CV Mean,Ticket Group CV Mean,Random-vs-Group Gap
0,Random Forest,0.8331,0.0221,0.8305,0.8238,0.0026
1,Extra Trees,0.8310,0.0201,0.8294,0.8215,0.0016
2,Logistic Regression,0.8278,0.0219,0.8204,0.8215,0.0074


# 17. Generate OOF Predictions for the Strongest Robust Model

We will identify the model with the highest average accuracy across the repeated random cross-validation experiments.

We will then generate out-of-fold probabilities using a fixed 5-fold split.

These predictions will be used for:

* Error analysis.
* Confusion matrix analysis.
* Identifying systematically difficult passenger groups.

We will not use these OOF predictions to create target-derived test features in this notebook.


In [28]:
best_model_name = (

    robust_results_df

    .iloc[0][
        "Model"
    ]
)


best_model = (

    models[
        best_model_name
    ]
)


best_pipeline = Pipeline(

    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            best_model
        )
    ]
)


oof_cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)


oof_probabilities = (

    cross_val_predict(

        best_pipeline,

        X,

        y,

        cv=oof_cv,

        method="predict_proba",

        n_jobs=-1

    )[:, 1]
)


oof_predictions = (

    oof_probabilities

    >= 0.5

).astype(int)


print(
    f"Model: {best_model_name}"
)


print(
    f"OOF Accuracy: "
    f"{accuracy_score(y, oof_predictions):.4f}"
)

Model: Random Forest
OOF Accuracy: 0.8339


In [29]:
error_analysis = (

    train_opt[

        [

            "PassengerId",

            "Name",

            "Sex",

            "Age",

            "Pclass",

            "Title",

            "FamilySize",

            "TicketGroupSize",

            "FamilyID_Raw",

            "TicketClean",

            "Survived"
        ]

    ]

    .copy()
)


error_analysis[

    "Predicted"

] = oof_predictions


error_analysis[

    "SurvivalProbability"

] = oof_probabilities


error_analysis[

    "Correct"

] = (

    error_analysis[
        "Survived"
    ]

    ==

    error_analysis[
        "Predicted"
    ]
)


print(
    "Correct predictions:"
)


print(

    error_analysis[
        "Correct"
    ]
    .value_counts()
)

Correct predictions:
Correct
True     743
False    148
Name: count, dtype: int64


In [30]:
error_by_group = (

    error_analysis

    .groupby(

        [
            "Sex",
            "Pclass"
        ]

    )[
        "Correct"
    ]

    .agg(

        [
            "count",
            "mean"
        ]
    )

    .reset_index()
)


error_by_group[

    "Error Rate"

] = (

    1

    -

    error_by_group[
        "mean"
    ]
)


error_by_group.round(4)

,Sex,Pclass,count,mean,Error Rate
0,female,1,94,0.9681,0.0319
1,female,2,76,0.9211,0.0789
2,female,3,144,0.6528,0.3472
3,male,1,122,0.6393,0.3607
4,male,2,108,0.9259,0.0741
5,male,3,347,0.8934,0.1066


# 18. Inspect Confident Model Errors

The most informative mistakes are often predictions where the model was highly confident but wrong.

Examples include:

* A passenger predicted to survive with 90% probability who did not survive.
* A passenger predicted to die with 90% probability who survived.

These cases may reveal patterns that the current features fail to capture.

We will calculate prediction confidence and inspect the strongest errors.


In [31]:
error_analysis[

    "Confidence"

] = np.where(

    error_analysis[
        "Predicted"
    ]
    == 1,

    error_analysis[
        "SurvivalProbability"
    ],

    1

    -

    error_analysis[
        "SurvivalProbability"
    ]
)


confident_errors = (

    error_analysis[

        error_analysis[
            "Correct"
        ]
        == False

    ]

    .sort_values(

        "Confidence",

        ascending=False
    )
)


confident_errors.head(
    20
)

,PassengerId,Name,Sex,Age,Pclass,Title,FamilySize,TicketGroupSize,FamilyID_Raw,TicketClean,Survived,Predicted,SurvivalProbability,Correct,Confidence
177,178,"Isham, Miss. Ann Elizabeth",female,50.0,1,Miss,1,1,Isham_1,PC17595,0,1,0.936320,False,0.936320
498,499,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0,1,Mrs,4,6,Allison_4,113781,0,1,0.915514,False,0.915514
444,445,"Johannesen-Bratthammer, Mr. Bernt",male,26.0,3,Mr,1,1,Johannesen-Bratthammer_1,65306,1,0,0.090190,False,0.909810
804,805,"Hedman, Mr. Oskar Arvid",male,27.0,3,Mr,1,1,Hedman_1,347089,1,0,0.091955,False,0.908045
36,37,"Mamee, Mr. Hanna",male,26.0,3,Mr,1,1,Mamee_1,2677,1,0,0.093524,False,0.906476
391,392,"Jansson, Mr. Carl Olof",male,21.0,3,Mr,1,1,Jansson_1,350034,1,0,0.093634,False,0.906366
338,339,"Dahl, Mr. Karl Edwart",male,45.0,3,Mr,1,1,Dahl_1,7598,1,0,0.096481,False,0.903519
510,511,"Daly, Mr. Eugene Patrick",male,29.0,3,Mr,1,1,Daly_1,382651,1,0,0.099162,False,0.900838
553,554,"Leeni, Mr. Fahim (""Philip Zenni"")",male,22.0,3,Mr,1,1,Leeni_1,2620,1,0,0.099507,False,0.900493
821,822,"Lulic, Mr. Nikola",male,27.0,3,Mr,1,1,Lulic_1,315098,1,0,0.099596,False,0.900404


# 19. Train the Strongest Robust Candidate

We will train the model with the strongest repeated cross-validation performance using the complete training dataset.

This model represents our Phase 10 competition candidate.

It is selected based on stability across multiple validation splits rather than performance on one specific cross-validation seed.


In [32]:
X_test = test_opt[
    features
].copy()


best_pipeline.fit(

    X,

    y
)


test_probabilities = (

    best_pipeline

    .predict_proba(

        X_test
    )[:, 1]
)


test_predictions = (

    test_probabilities

    >= 0.5

).astype(int)


print(
    "Test prediction distribution:"
)


print(

    pd.Series(
        test_predictions
    )

    .value_counts()

    .sort_index()
)

Test prediction distribution:
0    260
1    158
Name: count, dtype: int64


In [33]:
robust_submission = pd.DataFrame({

    "PassengerId":

        test_opt[
            "PassengerId"
        ]
        .astype(int),

    "Survived":

        test_predictions
})


assert len(
    robust_submission
) == len(
    test_raw
)


assert robust_submission[
    "PassengerId"
].is_unique


assert robust_submission[
    "Survived"
].isna().sum() == 0


assert set(

    robust_submission[
        "Survived"
    ].unique()

).issubset(
    {0, 1}
)


robust_submission_path = (

    submissions_dir

    /

    "submission_phase10_robust_model.csv"
)


robust_submission.to_csv(

    robust_submission_path,

    index=False
)


print(
    f"Saved:"
)


print(
    robust_submission_path
)

Saved:
../submissions/submission_phase10_robust_model.csv


In [34]:
gender_submission = pd.DataFrame({

    "PassengerId":

        test_opt[
            "PassengerId"
        ]
        .astype(int),

    "Survived":

        (
            test_opt[
                "Sex"
            ]
            == "female"
        )
        .astype(int)
})


gender_submission_path = (

    submissions_dir

    /

    "submission_phase10_gender_baseline.csv"
)


gender_submission.to_csv(

    gender_submission_path,

    index=False
)


print(
    gender_submission_path
)

../submissions/submission_phase10_gender_baseline.csv


In [35]:
full_validation_comparison.to_csv(

    results_dir

    /

    "phase10_validation_comparison.csv",

    index=False
)


error_analysis.to_csv(

    results_dir

    /

    "phase10_oof_error_analysis.csv",

    index=False
)


print(
    "Phase 10 diagnostics saved."
)

Phase 10 diagnostics saved.


# 20. How to Interpret Phase 10 Results

This notebook produces several different validation estimates.

They should not be interpreted identically.

---

## Repeated Stratified Cross-Validation

This measures how consistently a model performs across many random train-validation splits.

A strong model should have:

* High mean accuracy.
* Low standard deviation.
* Reasonable worst-case performance.

---

## Family Group Validation

This deliberately prevents members of the same identified family from appearing across training and validation folds.

A large performance drop may indicate that random cross-validation benefits from shared family patterns.

---

## Ticket Group Validation

This prevents passengers sharing a ticket from appearing across training and validation folds.

A large performance drop may indicate dependence on travel-group patterns that are distributed differently between training and unseen data.

---

# Important Interpretation

Group-aware validation is intentionally difficult.

Its score should not automatically be interpreted as the expected Kaggle score.

Instead, it acts as a diagnostic.

The most trustworthy models are those that perform reasonably well under:

* Repeated random validation.
* Family-aware validation.
* Ticket-aware validation.

We are looking for robustness across validation assumptions rather than maximizing one number.
